# Medallion Architecture — 02 Silver Dedup + Quarantine

Goal: understand deduplication as a Silver-layer pattern.

In `01_bronze_to_silver_quality`, we learned:

```text
BRONZE
 |
 v
SILVER QUALITY
 |
 +--> SILVER VALID
 |
 +--> QUARANTINE
```

In this notebook we focus on duplicate events:

```text
SILVER CANDIDATES
 |
 v
DEDUP RULE
 |
 +--> SILVER UNIQUE
 |
 +--> DUPLICATE QUARANTINE
```

Main idea:

```text
Duplicates are not silently dropped.
We select the winning row for Silver.
We keep duplicate rows separately for inspection.
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-silver-dedup-quarantine")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_medallion_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Silver candidate input

We simulate rows that already passed basic quality checks.

But some events arrive more than once.

```text
+----------+---------+------------+--------+---------------------+---------------------+
| event_id | user_id | event_type | amount | event_time          | ingestion_time      |
+----------+---------+------------+--------+---------------------+---------------------+
| e1       | u1      | purchase   | 10.0   | 2026-05-02 10:00:00 | 2026-05-02 10:01:00 |
| e2       | u1      | purchase   | 15.0   | 2026-05-02 10:05:00 | 2026-05-02 10:06:00 |
| e2       | u1      | purchase   | 15.0   | 2026-05-02 10:05:00 | 2026-05-02 10:07:00 | duplicate
| e3       | u2      | refund     | 5.0    | 2026-05-02 10:10:00 | 2026-05-02 10:11:00 |
| e3       | u2      | refund     | 7.0    | 2026-05-02 10:10:00 | 2026-05-02 10:12:00 | newer correction
+----------+---------+------------+--------+---------------------+---------------------+
```

For duplicates, we use this rule:

```text
For the same event_id, keep the row with the latest ingestion_time.
```

In [2]:
rows = [
    ("e1", "u1", "purchase", 10.0, "2026-05-02 10:00:00", "2026-05-02 10:01:00"),
    ("e2", "u1", "purchase", 15.0, "2026-05-02 10:05:00", "2026-05-02 10:06:00"),
    ("e2", "u1", "purchase", 15.0, "2026-05-02 10:05:00", "2026-05-02 10:07:00"),
    ("e3", "u2", "refund", 5.0, "2026-05-02 10:10:00", "2026-05-02 10:11:00"),
    ("e3", "u2", "refund", 7.0, "2026-05-02 10:10:00", "2026-05-02 10:12:00"),
]

silver_candidates = (
    spark.createDataFrame(
        rows,
        ["event_id", "user_id", "event_type", "amount", "event_time_raw", "ingestion_time_raw"]
    )
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
    .withColumn("ingestion_time", F.to_timestamp("ingestion_time_raw"))
    .drop("event_time_raw", "ingestion_time_raw")
)

silver_candidates.show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |
+--------+-------+----------+------+-------------------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|2026-05-02 10:01:00|
|e2      |u1     |purchase  |15.0  |2026-05-02 10:05:00|2026-05-02 10:06:00|
|e2      |u1     |purchase  |15.0  |2026-05-02 10:05:00|2026-05-02 10:07:00|
|e3      |u2     |refund    |5.0   |2026-05-02 10:10:00|2026-05-02 10:11:00|
|e3      |u2     |refund    |7.0   |2026-05-02 10:10:00|2026-05-02 10:12:00|
+--------+-------+----------+------+-------------------+-------------------+



## Step 2 — Detect duplicate groups

Before removing duplicates, make the duplicate situation visible.

```text
GROUP BY event_id
COUNT rows
```

If count > 1, that `event_id` has duplicates.

In [3]:
duplicate_groups = (
    silver_candidates
    .groupBy("event_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
)

duplicate_groups.show()

[Stage 2:>                                                          (0 + 2) / 2]

+--------+---------+
|event_id|row_count|
+--------+---------+
|      e2|        2|
|      e3|        2|
+--------+---------+



## Step 3 — Rank rows inside each event_id

We use a window function.

```text
PARTITION BY event_id
ORDER BY ingestion_time DESC
```

Then:

```text
rn = 1  -> winner
rn > 1  -> duplicate
```

In [4]:
dedup_window = (
    Window
    .partitionBy("event_id")
    .orderBy(F.col("ingestion_time").desc())
)

ranked = (
    silver_candidates
    .withColumn("dedup_rank", F.row_number().over(dedup_window))
    .withColumn("duplicate_count", F.count("*").over(Window.partitionBy("event_id")))
)

ranked.orderBy("event_id", "dedup_rank").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+----------+---------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |dedup_rank|duplicate_count|
+--------+-------+----------+------+-------------------+-------------------+----------+---------------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|2026-05-02 10:01:00|1         |1              |
|e2      |u1     |purchase  |15.0  |2026-05-02 10:05:00|2026-05-02 10:07:00|1         |2              |
|e2      |u1     |purchase  |15.0  |2026-05-02 10:05:00|2026-05-02 10:06:00|2         |2              |
|e3      |u2     |refund    |7.0   |2026-05-02 10:10:00|2026-05-02 10:12:00|1         |2              |
|e3      |u2     |refund    |5.0   |2026-05-02 10:10:00|2026-05-02 10:11:00|2         |2              |
+--------+-------+----------+------+-------------------+-------------------+----------+---------------+



## Step 4 — Silver unique rows

Silver keeps one business row per `event_id`.

```text
dedup_rank = 1
```

This is the clean output for downstream Gold tables.

In [5]:
silver_unique = (
    ranked
    .filter(F.col("dedup_rank") == 1)
    .drop("dedup_rank", "duplicate_count")
)

silver_unique.orderBy("event_id").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |
+--------+-------+----------+------+-------------------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|2026-05-02 10:01:00|
|e2      |u1     |purchase  |15.0  |2026-05-02 10:05:00|2026-05-02 10:07:00|
|e3      |u2     |refund    |7.0   |2026-05-02 10:10:00|2026-05-02 10:12:00|
+--------+-------+----------+------+-------------------+-------------------+



## Step 5 — Duplicate quarantine

Rows with `dedup_rank > 1` are not used in Silver output.

But they are still useful.

```text
duplicate quarantine = rows that lost the dedup rule
```

We keep them with an `error_reason`.

In [6]:
duplicate_quarantine = (
    ranked
    .filter(F.col("dedup_rank") > 1)
    .withColumn("error_reason", F.lit("duplicate_event_id"))
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "event_time",
        "ingestion_time",
        "dedup_rank",
        "duplicate_count",
        "error_reason"
    )
)

duplicate_quarantine.orderBy("event_id", "dedup_rank").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+----------+---------------+------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |dedup_rank|duplicate_count|error_reason      |
+--------+-------+----------+------+-------------------+-------------------+----------+---------------+------------------+
|e2      |u1     |purchase  |15.0  |2026-05-02 10:05:00|2026-05-02 10:06:00|2         |2              |duplicate_event_id|
|e3      |u2     |refund    |5.0   |2026-05-02 10:10:00|2026-05-02 10:11:00|2         |2              |duplicate_event_id|
+--------+-------+----------+------+-------------------+-------------------+----------+---------------+------------------+



## Step 6 — Compare before and after

A good habit:

```text
input rows
unique Silver rows
duplicate quarantine rows
```

In [7]:
summary = spark.createDataFrame(
    [
        ("silver_candidate_rows", silver_candidates.count()),
        ("silver_unique_rows", silver_unique.count()),
        ("duplicate_quarantine_rows", duplicate_quarantine.count()),
    ],
    ["metric", "value"]
)

summary.show(truncate=False)

+-------------------------+-----+
|metric                   |value|
+-------------------------+-----+
|silver_candidate_rows    |5    |
|silver_unique_rows       |3    |
|duplicate_quarantine_rows|2    |
+-------------------------+-----+



## Step 7 — Why not just dropDuplicates?

Spark has:

```python
df.dropDuplicates(["event_id"])
```

But it does not clearly explain which row wins.

For teaching and production-like pipelines, this is better:

```text
1. define dedup key
2. define ordering rule
3. rank rows
4. keep winner
5. quarantine losers
```

That makes the pipeline explainable.

In [8]:
drop_duplicates_example = silver_candidates.dropDuplicates(["event_id"])

drop_duplicates_example.orderBy("event_id").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |
+--------+-------+----------+------+-------------------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|2026-05-02 10:01:00|
|e2      |u1     |purchase  |15.0  |2026-05-02 10:05:00|2026-05-02 10:06:00|
|e3      |u2     |refund    |5.0   |2026-05-02 10:10:00|2026-05-02 10:11:00|
+--------+-------+----------+------+-------------------+-------------------+



## Final mental model

```text
SILVER CANDIDATES
 |
 v
DEDUP RANKING
- partition by event_id
- order by ingestion_time desc
 |
 +--> SILVER UNIQUE
 |    - dedup_rank = 1
 |    - one row per event_id
 |
 +--> DUPLICATE QUARANTINE
      - dedup_rank > 1
      - error_reason = duplicate_event_id
```

This is the Silver deduplication pattern.

In [9]:
spark.stop()